In [ ]:
!pip install \
    cohere \
    langchain \
    tiktoken \
    pinecone-client \
    langchain-openai \
    langchain-pinecone \
    sentence-transformers \
    gradio

In [ ]:
from google.colab import userdata, drive
from langchain_pinecone import PineconeVectorStore as LangChainPinecone
from langchain_openai import ChatOpenAI as LangChainChatOpenAI
from langchain.chains import RetrievalQA
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore as LangChainPinecone
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import re

In [ ]:
from IPython.display import HTML, display
def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
class ChapterData:

  def __init__(self):
    self.overviews = self.load_data('snippets/overview_details.txt', '\n-----\n\n')
    self.languages = self.load_data('snippets/languages_per_chapter.txt', '\n' )

  def load_data(self, filename, splitter):
    d = dict()
    with open(f'/content/drive/MyDrive/WALS/{filename}', encoding='utf-8') as f:
      data = f.read()
    chunks = data.split(splitter)
    for chunk in chunks:
      try:
        chapter = chunk.split(' ', maxsplit=2)[:2][1]
        d[chapter] = chunk
      except IndexError:
        pass #ignore blanks at end of file
    return d

In [ ]:
class PinkFloyd:

  def __init__(self,
               index,
               embeddings,
               embed_function,
               llm,
               turns_in_context,
               vector_match_threshold):
    self.index = index
    self.embeddings = embeddings
    self.embed_function = embed_function
    self.llm = llm
    self.turns_in_context = turns_in_context
    self.threshold = vector_match_threshold
    self.load_prompts()
    self.reg_ex = re.compile(r'(?i)chapter \d+')
    self.chapter_data = ChapterData()

  def load_prompts(self):
    self.prompts = dict()
    path = '/content/drive/MyDrive/WALS/prompts'
    with open(f'{path}/system_prompt.txt') as f:
      self.prompts['system'] = f.read()
    with open(f'{path}/chapter_text_prompt.txt') as f:
      self.prompts['chapter_text'] = f.read()
    with open(f'{path}/map_values_prompt.txt') as f:
      self.prompts['map_values'] = f.read()
    with open(f'{path}/typology_details_prompt.txt') as f:
      self.prompts['typology_details'] = f.read()
    with open(f'{path}/chapter_details_prompt.txt') as f:
      self.prompts['chapter_details'] = f.read()

  def query_pinecone(self, message, namespace, k=5):
        results = list()
        if namespace == 'chapters':
          results.append(self.prompts['chapter_text'])
        elif namespace == 'maps':
          results.append(self.prompts['map_values'])
        elif namespace == 'typology':
          results.append(self.prompts['typology_details'])
        elif namespace == 'snippets':
          results.append('Here are some quick language facts that may be relevant')
        elif namespace == 'authors':
          results.append('Here is some authorship information')

        message_embeddings = self.embed_function(self.embeddings, message)
        pinecone_query = self.index.query(vector=message_embeddings,
                                top_k=k,
                                namespace=namespace,
                                include_metadata=True)
        #Loop through the results and break when either of these is true:
        #a) We have collected a number of documents equal to self.top_k
        #b) We have hit documents with a score below self.threshold, and we return fewer than self.top_k document
        results.extend(self.filter_top_k(pinecone_query, k))
        if len(results) == 1:
          #there's only a preamble, we matched no documents, return nothing
          results = []
        return results

  def filter_top_k(self, query, k):
    results = list()
    for result in query['matches']:
      if len(results) == k:
        break
        #we've hit our top_k
      if result['score'] > self.threshold:
        results.append('- ' + result['metadata']['text'])
      else:
        break
        #results are ordered by score, so if one doesn't pass the threshold none of the rest will
    return results


  def check_for_chapter_references(self, message, k = 5):
    results = list()
    results.append(self.prompts['chapter_details'])
    chapter_mentions = self.reg_ex.findall(message)
    if chapter_mentions is not None:
      for chapter in chapter_mentions:
        number = chapter.split(' ')[-1]
        results.append('- '+self.chapter_data.overviews[number])
        message_embeddings = self.embed_function(self.embeddings, message)
        summary_query = self.index.query(
                            vector=message_embeddings,
                            top_k=1,
                            filter={"chapter": number},
                            namespace='summaries',
                            include_metadata=True)
        for q in summary_query['matches']:
          summary = q['metadata']['text']
        results.append(summary)
    if len(results) == 1:
      results = []
    return results

  def process_message(self, message):
    extras = list()

    #User may frame a question as "what does WALS say about..."
    #the model often returns no results for this phrasing, possibly because the term 'WALS' is rarely mentioned in WALS
    #simplying swaping out for 'you' e.g. 'what does you say about..." returns much better results (even with the grammatical error)
    message = message.replace('WALS', 'you')

    #Check various vector spaces for good matches
    extras.extend(self.query_pinecone(message, namespace='chapters', k=10)) #raw chapter text
    extras.extend(self.query_pinecone(message, namespace='typology', k=3)) #data extracted from tables
    extras.extend(self.query_pinecone(message, namespace='maps', k=1)) #map values from chapter text
    extras.extend(self.query_pinecone(message, namespace='snippets', k=1)) #summaries or other 'synthetic' data files
    extras.extend(self.query_pinecone(message, namespace='authors', k=1)) #author names and which chapters they wrote

    #Next check for references to specific chapters and get a summary of that chapter
    extras.extend(self.check_for_chapter_references(message))

    if extras:
      extras = '\n'.join(extras)
    else:
      extras = self.prompts['system']
    return extras

  def construct_context(self, history):
    if not history:
      context_message = ['This is the first turn of your interaction with the user, there is no prior conversational history\n']
    else:
      context_message = ['Here\'s some recent history of the conversation for context:\n']
      for turn in history[-self.turns_in_context:]:
        context_message.append(f'You said: {turn[0]}\n')
        context_message.append(f'User said: {turn[1]}\n', )
    context_message.append('----------\n')
    context_message = ' '.join(context_message)
    return context_message

In [ ]:
class WALSBuilder:

    def __init__(self, open_ai_key, pinecone_key, text_embedding_model='ada', pinecone_index='starter-index', turns_in_context=3, vector_match_threshold=0.85):
        self.index = self.setup_index(api_key=pinecone_key, index_name=pinecone_index)
        self.embeddings, self.embed_function = self.setup_embeddings(model=text_embedding_model, api_key=open_ai_key)
        self.llm = self.setup_llm(api_key=open_ai_key)
        self.turns_in_context = turns_in_context
        self.threshold = vector_match_threshold
        self.agent = PinkFloyd(self.index, self.embeddings, self.embed_function, self.llm, self.turns_in_context, self.threshold)

    def setup_index(self, api_key, index_name):
      pc = Pinecone(api_key=api_key)
      index = pc.Index(index_name)
      return index

    def ada_embeddings(self, model, text):
      embeds = model.embed_documents(text)
      return embeds

    def mpnet_embeddings(self, model, text):
      embeds = model.encode(text).tolist()
      return embeds

    def setup_embeddings(self,model, api_key=None):
      if model == 'ada':
        embeds = LangChainOpenAIEmbeddings(openai_api_key=api_key, model='text-embedding-ada-002')
        embed_function = self.ada_embeddings
      elif model == 'mpnet':
        embeds = SentenceTransformer('all-mpnet-base-v2')
        embed_function = self.mpnet_embeddings
        return embeds, embed_function

    def setup_llm(self, api_key, model_name='gpt-3.5-turbo', temperature=0.2):
      llm = LangChainChatOpenAI(
            openai_api_key=api_key,
            model_name=model_name,
            temperature=temperature)
      return llm

    def TalkToWALS(self, query):
        extras = self.agent.process_message(query)
        message = '\n'.join(['Please help the user with this query:', query])
        send_to_llm = ''.join([self.prompts['system'], extras, message])

        response = self.agent.llm.invoke(send_to_llm)
        return response.content

    def refresh_agent(self):
      self.agent = PinkFloyd(self.index, self.embeddings, self.embed_function, self.llm, self.turns_in_context, self.threshold)

    def interact(self, message, history, show_prompt=False):
        #Add context about previous interactions
        context = self.agent.construct_context(history)

        #Look up additional information about the message
        #This could include information such as authorship, typology data, or lists of languages in a chapter
        #It's stored and searched separately from the main WALS chapter data because it 'pollutes' the Pinecone search results
        extras = self.agent.process_message(message)

        #Make a basic instruction from the user's query
        instruction = '\n'.join(['\nGiven all this context, please help the user with this query:', message])

        #Glue it all together and send to the LLM
        send_to_llm = '\n'.join([self.agent.prompts['system'], context, extras, instruction])
        if show_prompt:
          print(send_to_llm)
        response = self.agent.llm.invoke(send_to_llm)
        text = response.content
        return text

In [ ]:
class BuildingInspector:

  def __init__(self,
               golden,
               model_response,
               openai_api_key=userdata.get('PINECONE_TOKEN'),
               text_embedding_model='ada',
               pinecone_index='sentence-embeds'):
    self.goldens = goldens
    self.llm = self.setup_llm(api_key=openai_api_key)
    self.embeddings = self.setup_embeddings(api_key=openai_api_key, model='ada')
    path = ''
    with open(f'/content/drive/MyDrive/WALS/prompts/evaluation_prompt.txt') as f:
      self.prompt = f.read()
    self.evaluate(golden, model_response)

  def setup_llm(self, api_key, model_name='gpt-3.5-turbo', temperature=0.2):
    llm = LangChainChatOpenAI(
        openai_api_key=api_key,
        model_name=model_name,
        temperature=temperature)
    return llm

  def setup_embeddings(self, model, api_key=None):
    if model == 'ada':
      embeds = LangChainOpenAIEmbeddings(openai_api_key=api_key, model='text-embedding-ada-002')
      embed_function = self.ada_embeddings
    elif model == 'mpnet':
      embeds = SentenceTransformer('all-mpnet-base-v2')
      embed_function = self.mpnet_embeddings
      return embeds, embed_function

  def evaluate(self, model_response, golden_response):
    llm_prompt = f'{self.prompt}\nTarget:{golden_response}\nTest:{model_response} '
    response = self.agent.llm.invoke(send_to_llm)
    text = response.content

In [ ]:
wals = WALSBuilder(userdata.get('OPENAI_KEY'),
                    userdata.get('PINECONE_TOKEN'),
                   text_embedding_model='mpnet',
                   pinecone_index='sentence-embeds')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#drive.flush_and_unmount()

In [ ]:
#drive.mount('/content/drive')

In [ ]:
#wals.refresh_agent()
wals.interact('Are there any languages with five or more genders?', [])

"I'm sorry, but I couldn't find specific information on languages with five or more genders in the World Atlas of Language Structures. However, there are languages with multiple gender systems, so it's possible that some languages may have five or more genders. If you have any other questions or if there's a different topic you'd like to explore, feel free to ask!"

In [ ]:
import gradio as gr

In [ ]:
ui = gr.ChatInterface(wals.interact,
                 textbox=gr.Textbox(placeholder="Ask me about the World Atlas of Language Structures"),
                 title="Talking To WALS",
                 description="Ask me about the World Atlas of Language Structures",
                 examples=["What is chapter 17 about?",
                           "Which chapters did Ian Maddieson contribute to?",
                           "Tell me about the velar nasal in Siberian languages",
                           "What does WALS say about adnominals in Ingush?"],
                 )

ui.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://1ce68c90fc14da4312.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1ce68c90fc14da4312.gradio.live


In [ ]:
# !wget https://raw.githubusercontent.com/jsmackie/TalkingToWALS/preamble_types/wals.py -O wals.py
# !wget https://raw.githubusercontent.com/jsmackie/TalkingToWALS/mainline/HidingBehindWALS.py -O HidingBehindWALS.py
# import wals